In [ ]:
import pyspark

In [ ]:
pyspark.__version__


'3.3.0'

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession \
    .builder \
    .appName('corona Data') \
    .getOrCreate()

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving corona_tweets.csv to corona_tweets (2).csv


In [ ]:
import io
df = (spark.read
          .format("csv")
          .option('header', 'true')
          .load("corona_tweets.csv"))

In [ ]:
df.show()

+--------------------+--------------------+--------------------+--------------------+--------------------+---------+
|            UserName|          ScreenName|            Location|             TweetAt|       OriginalTweet|Sentiment|
+--------------------+--------------------+--------------------+--------------------+--------------------+---------+
|                3799|               48751|              London|          16-03-2020|@MeNyrbie @Phil_G...|  Neutral|
|                3800|               48752|                  UK|          16-03-2020|advice Talk to yo...| Positive|
|                3801|               48753|           Vagabonds|          16-03-2020|Coronavirus Austr...| Positive|
|                3802|               48754|                null|          16-03-2020|My food stock is ...|     null|
|              PLEASE|         don't panic| THERE WILL BE EN...|                null|                null|     null|
|           Stay calm|          stay safe.|                null|

In [ ]:
pd.DataFrame(df.take(5), columns=df.columns)

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,None,16-03-2020,My food stock is not the only one which is emp...,None
4,PLEASE,don't panic,THERE WILL BE ENOUGH FOOD FOR EVERYONE if you...,None,None,None


In [ ]:
df.printSchema()

root
 |-- UserName: string (nullable = true)
 |-- ScreenName: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- TweetAt: string (nullable = true)
 |-- OriginalTweet: string (nullable = true)
 |-- Sentiment: string (nullable = true)



In [ ]:
data = df
imputedf_pandas = data.toPandas()

In [ ]:
imputedf_pandas

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,None,16-03-2020,My food stock is not the only one which is emp...,None
4,PLEASE,don't panic,THERE WILL BE ENOUGH FOOD FOR EVERYONE if you...,None,None,None
...,...,...,...,...,...,...
68041,44952,89904,None,14-04-2020,Response to complaint not provided citing COVI...,Extremely Negative
68042,44953,89905,None,14-04-2020,You know its getting tough when @KameronWilds...,Positive
68043,44954,89906,None,14-04-2020,Is it wrong that the smell of hand sanitizer i...,None
68044,"#coronavirus #COVID19 #coronavirus""",Neutral,None,None,None,None


In [ ]:
imputedf_pandas.columns

Index(['UserName', 'ScreenName', 'Location', 'TweetAt', 'OriginalTweet',
       'Sentiment'],
      dtype='object')

In [ ]:
imputedf_pandas.TweetAt.value_counts()

20-03-2020                        3447
19-03-2020                        3215
25-03-2020                        2978
18-03-2020                        2742
21-03-2020                        2652
                                  ... 
 rice...."                           1
 and it wont stop.                  1
 HUL https://t.co/ttyJY9IgdW"        1
 #hoarders"                          1
722.20 earlier in the session.       1
Name: TweetAt, Length: 370, dtype: int64

In [ ]:
imputedf_pandas['Location'].value_counts()

London                                                                                                                               540
United States                                                                                                                        528
London, England                                                                                                                      520
New York, NY                                                                                                                         395
Washington, DC                                                                                                                       373
                                                                                                                                    ... 
Highland, CA                                                                                                                           1
 I'm unable to book any slots at all from

In [ ]:
imputedf_pandas['Sentiment'].value_counts()

Positive                                                                                       7718
Negative                                                                                       6857
Neutral                                                                                        5224
Extremely Positive                                                                             4412
Extremely Negative                                                                             3751
                                                                                               ... 
 the fall in oil and stock prices has widened bond spreads and put pressure on the Naira.""       1
 a stock market nose-dive                                                                         1
 fast food workers                                                                                1
 lol..."                                                                                          1


In [ ]:
loc_analysis = pd.DataFrame(imputedf_pandas['Location'].value_counts().sort_values(ascending=False))
loc_analysis = loc_analysis.rename(columns={'Location':'count'})

In [ ]:
import plotly.graph_objects as go

In [ ]:
data = {
   "values": loc_analysis['count'][:15],
   "labels": loc_analysis.index[:15],
   "domain": {"column": 0},
   "name": "Location Name",
   "hoverinfo":"label+percent+name",
   "hole": .4,
   "type": "pie"
}
layout = go.Layout(title="<b>Ratio on Location</b>", legend=dict(x=0.1, y=1.1, orientation="h"))

data = [data]
fig = go.Figure(data = data, layout = layout)
fig.update_layout(title_x=0.5)
fig.show()

## Data Preprocessing

In [ ]:
# write function for removing @user
def remove_pattern(input_txt, pattern):
    r = re.findall(pattern, str(input_txt))
    for i in r:
        input_txt = re.sub(i,'',input_txt)
    return input_txt

In [ ]:
# create new column with removed @user
imputedf_pandas['Tweet'] = np.vectorize(remove_pattern)(imputedf_pandas['OriginalTweet'], '@[\w]*')

In [ ]:
imputedf_pandas.head(2)

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment,Tweet
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral,https://t.co/iFz9FAn2Pa and https://t.co/xX...
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive,advice Talk to your neighbours family to excha...


## Removed HTTP and urls from tweet

In [ ]:
imputedf_pandas['Tweet'] = imputedf_pandas['Tweet'].apply(lambda x: re.split('https:\/\/.*', str(x))[0])

In [ ]:
imputedf_pandas.head(3)

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment,Tweet
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral,
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive,advice Talk to your neighbours family to excha...
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive,Coronavirus Australia: Woolworths to give elde...


In [ ]:
imputedf_pandas['Tweet'] = imputedf_pandas['Tweet'].str.replace('[^a-zA-Z#]+',' ')

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: FutureWarning:

The default value of regex will change from True to False in a future version.



In [ ]:
# remove short words
imputedf_pandas['Tweet'] = imputedf_pandas['Tweet'].apply(lambda x: ' '.join([w for w in x.split() if len(w) > 2]))

In [ ]:
# create new variable tokenized tweet 
tokenized_tweet = imputedf_pandas['Tweet'].apply(lambda x: x.split())

In [ ]:
from nltk.stem.porter import *
stemmer = PorterStemmer()

In [ ]:
# apply stemmer for tokenized_tweet
tokenized_tweet = tokenized_tweet.apply(lambda x: [stemmer.stem(i) for i in x])

In [ ]:
# join tokens into one sentence
for i in range(len(tokenized_tweet)):
    tokenized_tweet[i] = ' '.join(tokenized_tweet[i])
# change df['Tweet'] to tokenized_tweet

In [ ]:
print(tokenized_tweet)

0                                                         
1        advic talk your neighbour famili exchang phone...
2        coronaviru australia woolworth give elderli di...
3                  food stock not the onli one which empti
4                                                     none
                               ...                        
68041    respons complaint not provid cite covid relat ...
68042    you know get tough when ration toilet paper #c...
68043           wrong that the smell hand sanit start turn
68044                                                 none
68045    well new use rift are go for amazon although t...
Name: Tweet, Length: 68046, dtype: object


In [ ]:
imputedf_pandas['Tweet']  = tokenized_tweet

In [ ]:
new_df = imputedf_pandas[['Tweet','Sentiment']]

In [ ]:
new_df.head()

,Tweet,Sentiment
0,,Neutral
1,advic talk your neighbour famili exchang phone...,Positive
2,coronaviru australia woolworth give elderli di...,Positive
3,food stock not the onli one which empti,None
4,none,None


In [ ]:
new_df.isnull().sum()

Tweet            0
Sentiment    39429
dtype: int64

In [ ]:
new_data = new_df.dropna()

In [ ]:
new_data

,Tweet,Sentiment
0,,Neutral
1,advic talk your neighbour famili exchang phone...,Positive
2,coronaviru australia woolworth give elderli di...,Positive
10,news the region first confirm covid case came ...,Positive
11,cashier groceri store wa share hi insight #cov...,Positive
...,...,...
68039,you are definit man feel like thi fall when ar...,Extremely Positive
68040,airlin pilot offer stock supermarket shelv #nz...,Neutral
68041,respons complaint not provid cite covid relat ...,Extremely Negative
68042,you know get tough when ration toilet paper #c...,Positive


In [ ]:
from sklearn.model_selection import train_test_split
# x = new_data['Tweet'].tolist()
# y = new_data['Sentiment'].tolist()
train,valid = train_test_split(new_data,test_size = 0.2,random_state=0)
print("train shape : ", train.shape)
print("valid shape : ", valid.shape)

train shape :  (22893, 2)
valid shape :  (5724, 2)


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop = list(stopwords.words('english'))
vectorizer = CountVectorizer(ngram_range=(1,1),decode_error = 'replace',stop_words = stop)

X_train = vectorizer.fit_transform(train.Tweet.values)
X_valid = vectorizer.transform(valid.Tweet.values)

y_train = train.Sentiment.values
y_valid = valid.Sentiment.values

print("X_train.shape : ", X_train.shape)
print("X_valid.shape : ", X_valid.shape)
print("y_train.shape : ", y_train.shape)
print("y_valid.shape : ", y_valid.shape)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


X_train.shape :  (22893, 20063)
X_valid.shape :  (5724, 20063)
y_train.shape :  (22893,)
y_valid.shape :  (5724,)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,classification_report
clf = DecisionTreeClassifier(random_state=0,criterion="gini")
clf.fit(X_train,y_train)

DecisionTreeClassifier(random_state=0)

In [ ]:
prediction = clf.predict(X_valid)
accuracy = accuracy_score(y_valid,prediction)
print("training accuracy Score    : ",clf.score(X_train,y_train))
print("Validation accuracy Score : ",accuracy )

training accuracy Score    :  0.9965054820250732
Validation accuracy Score :  0.47379454926624737
